In [9]:
%pip install pandas numpy TensorFlow Pytorch graphdatascience fastparquet

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached pytorch-1.0.2.tar.gz (689 bytes)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.met

  error: subprocess-exited-with-error
  
  × Building wheel for Pytorch (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [37 lines of output]
      Traceback (most recent call last):
        File "c:\Users\zunde\anaconda3\envs\TFM\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "c:\Users\zunde\anaconda3\envs\TFM\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\zunde\anaconda3\envs\TFM\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 280, in build_wheel
          return _build_backend().build_wheel(
                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
              wheel_directory, config_settings, metadata_directory
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [10]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
import time

# Configuración
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "1029,Ferni")
DB_NAME = "TFM"
FILE_PATH = "part-00000-8b838a85-76eb-4896-a0b6-2fc425e828c2-c000.snappy.parquet"

# Leer el archivo
df = pd.read_parquet(FILE_PATH)

# Quedarnos solo con las columnas necesarias
columnas = [
    'conn_state', 'duration', 'src_ip_zeek', 'src_port_zeek', 'dest_ip_zeek', 
    'dest_port_zeek', 'local_orig', 'local_resp', 'missed_bytes', 'orig_bytes', 
    'orig_pkts', 'proto', 'resp_bytes', 'resp_pkts', 'service', 'ts', 'uid', 
    'label_tactic', 'label_binary'
]
df = df[columnas]

df['label_tactic'] = df['label_tactic'].fillna('Benign')
df = df.replace({np.nan: None})

def upload_to_neo4j(tx, batch):
    query = """
    UNWIND $rows AS row
    
    MERGE (src:IP {address: row.src_ip_zeek})
    ON CREATE SET src.is_local = row.local_orig
    
    MERGE (dst:IP {address: row.dest_ip_zeek})
    ON CREATE SET dst.is_local = row.local_resp
    
    CREATE (src)-[r:NETWORK_CONNECTION]->(dst)
    SET r.uid = row.uid,
        
        // --- OPCIÓN 1: TIEMPO HISTÓRICO ORIGINAL ---
        // r.ts = toFloat(row.ts), 
        
        // --- OPCIÓN 2: SIMULADOR EN TIEMPO REAL PARA PROBAR REACT ---
        // (Usa el reloj actual de Neo4j en segundos)
        r.ts = timestamp() / 1000.0,
        
        r.duration = toFloat(row.duration),
        r.orig_bytes = toInteger(row.orig_bytes),
        r.resp_bytes = toInteger(row.resp_bytes),
        r.missed_bytes = toInteger(row.missed_bytes),
        r.orig_pkts = toInteger(row.orig_pkts),
        r.resp_pkts = toInteger(row.resp_pkts),
        r.src_port = toInteger(row.src_port_zeek),
        r.dest_port = toInteger(row.dest_port_zeek),
        r.proto = row.proto,
        r.service = row.service,
        r.conn_state = row.conn_state,
        r.label_tactic = row.label_tactic,
        r.label_binary = toString(row.label_binary)
    """
    tx.run(query, rows=batch)

# Conexión a la base de datos
driver = GraphDatabase.driver(URI, auth=AUTH)

print(f"Preparando la base de datos '{DB_NAME}'...")
with driver.session(database=DB_NAME) as session:
    # 1. Limpiar base de datos
    print("Borrando nodos y relaciones...")
    session.run("MATCH (n) DETACH DELETE n")
    
    # 2. Borrar índice viejo (buena práctica al resetear)
    session.run("DROP INDEX connection_ts_idx IF EXISTS")
    
    # 3. Crear el índice nuevo para el campo ts
    print("Creando índice para optimizar consultas en tiempo real...")
    session.run("CREATE INDEX connection_ts_idx IF NOT EXISTS FOR ()-[r:NETWORK_CONNECTION]-() ON (r.ts)")
    
print("Base de datos limpia e indexada.")

batch_size = 10
sleep_time = 0.5 # Pausa en segundos entre cada lote

with driver.session(database=DB_NAME) as session:
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size].to_dict('records')
        session.execute_write(upload_to_neo4j, batch)
        print(f"Cargados {min(i + batch_size, len(df))} de {len(df)} registros...")
        
        # Hacemos que el script "duerma" para no saturar Neo4j ni el Frontend
        time.sleep(sleep_time)

driver.close()
print("¡Carga finalizada con éxito!")

Preparando la base de datos 'TFM'...
Borrando nodos y relaciones...
Creando índice para optimizar consultas en tiempo real...
Base de datos limpia e indexada.
Cargados 10 de 255443 registros...
Cargados 20 de 255443 registros...
Cargados 30 de 255443 registros...
Cargados 40 de 255443 registros...
Cargados 50 de 255443 registros...
Cargados 60 de 255443 registros...
Cargados 70 de 255443 registros...
Cargados 80 de 255443 registros...
Cargados 90 de 255443 registros...
Cargados 100 de 255443 registros...
Cargados 110 de 255443 registros...
Cargados 120 de 255443 registros...
Cargados 130 de 255443 registros...
Cargados 140 de 255443 registros...
Cargados 150 de 255443 registros...
Cargados 160 de 255443 registros...
Cargados 170 de 255443 registros...
Cargados 180 de 255443 registros...
Cargados 190 de 255443 registros...
Cargados 200 de 255443 registros...
Cargados 210 de 255443 registros...
Cargados 220 de 255443 registros...
Cargados 230 de 255443 registros...
Cargados 240 de 255443

KeyboardInterrupt: 

In [ ]:

import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# Configuración
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "1029,Ferni")
DB_NAME = "TFM"
FILE_PATH = "part-00000-8b838a85-76eb-4896-a0b6-2fc425e828c2-c000.snappy.parquet"

# Leer el archivo
df = pd.read_parquet(FILE_PATH)

# Quedarnos solo con las columnas necesarias (opcional, pero ahorra memoria)
columnas = [
    'conn_state', 'duration', 'src_ip_zeek', 'src_port_zeek', 'dest_ip_zeek', 
    'dest_port_zeek', 'local_orig', 'local_resp', 'missed_bytes', 'orig_bytes', 
    'orig_pkts', 'proto', 'resp_bytes', 'resp_pkts', 'service', 'ts', 'uid', 
    'label_tactic', 'label_binary'
]
df = df[columnas]

df['label_tactic'] = df['label_tactic'].fillna('Benign')

df = df.replace({np.nan: None})

def upload_to_neo4j(tx, batch):
    query = """
    UNWIND $rows AS row
    
    MERGE (src:IP {address: row.src_ip_zeek})
    ON CREATE SET src.is_local = row.local_orig
    
    MERGE (dst:IP {address: row.dest_ip_zeek})
    ON CREATE SET dst.is_local = row.local_resp
    
    MERGE (p_dst:Port {number: toInteger(row.dest_port_zeek)})
    
    CREATE (src)-[r:NETWORK_CONNECTION]->(dst)
    SET r.uid = row.uid,
        r.ts = toFloat(row.ts),
        r.duration = toFloat(row.duration),
        r.orig_bytes = toInteger(row.orig_bytes),
        r.resp_bytes = toInteger(row.resp_bytes),
        r.missed_bytes = toInteger(row.missed_bytes),
        r.orig_pkts = toInteger(row.orig_pkts),
        r.resp_pkts = toInteger(row.resp_pkts),
        r.src_port = toInteger(row.src_port_zeek),
        r.dest_port = toInteger(row.dest_port_zeek),
        r.proto = row.proto,
        r.service = row.service,
        r.conn_state = row.conn_state,
        r.label_tactic = row.label_tactic,
        r.label_binary = toString(row.label_binary)
        
    MERGE (dst)-[:LISTENS_ON]->(p_dst)
    """
    tx.run(query, rows=batch)

# Conexión a la base de datos
driver = GraphDatabase.driver(URI, auth=AUTH)

print(f"Borrando todos los elementos antiguos de la base de datos '{DB_NAME}'...")
with driver.session(database=DB_NAME) as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Base de datos limpia.")

# Carga por lotes (batching) para mayor velocidad
batch_size = 50
with driver.session(database=DB_NAME) as session:
    # Convertimos el DataFrame a diccionario solo al momento de iterar
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size].to_dict('records')
        session.execute_write(upload_to_neo4j, batch)
        print(f"Cargados {min(i + batch_size, len(df))} de {len(df)} registros...")

driver.close()
print("¡Carga finalizada con éxito!")

Borrando todos los elementos antiguos de la base de datos 'TFM'...
Base de datos limpia.
Cargados 50 de 255443 registros...
Cargados 100 de 255443 registros...
Cargados 150 de 255443 registros...
Cargados 200 de 255443 registros...
Cargados 250 de 255443 registros...
Cargados 300 de 255443 registros...
Cargados 350 de 255443 registros...
Cargados 400 de 255443 registros...
Cargados 450 de 255443 registros...
Cargados 500 de 255443 registros...
Cargados 550 de 255443 registros...
Cargados 600 de 255443 registros...
Cargados 650 de 255443 registros...
Cargados 700 de 255443 registros...
Cargados 750 de 255443 registros...
Cargados 800 de 255443 registros...
Cargados 850 de 255443 registros...
Cargados 900 de 255443 registros...
Cargados 950 de 255443 registros...
Cargados 1000 de 255443 registros...
Cargados 1050 de 255443 registros...
Cargados 1100 de 255443 registros...
Cargados 1150 de 255443 registros...
Cargados 1200 de 255443 registros...
Cargados 1250 de 255443 registros...
Cargad